# Assignment 1 - Advanced Matching and Weak-Supervision Refinement

This notebook builds a multi-signal matching pipeline for Assignment 1. PJMISO/Market is the reference dataset. PANO and DAYZER are candidate match sources.

Generated outputs are written to `solution/assignment_1/execution_outputs/`. Embedding similarity is included as an optional disabled placeholder with `USE_EMBEDDINGS = False`.

In [ ]:
from collections import defaultdict
from pathlib import Path
import re

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

## Configuration

In [ ]:
TOP_K_PER_SOURCE = 25
MAX_CANDIDATES_TO_SCORE = 250
LOW_SCORE_REVIEW_THRESHOLD = 0.75
CLOSE_MARGIN_THRESHOLD = 0.05
MIN_ACCEPT_SCORE = 0.50
HIGH_CONFIDENCE_THRESHOLD = 0.85
MEDIUM_CONFIDENCE_THRESHOLD = 0.70
RANDOM_AUDIT_SAMPLE_SIZE = 100
MANUAL_LABEL_SAMPLE_SIZE_PER_BUCKET = 100
MIN_LABELS_PER_CLASS = 100
CLASSIFIER_MODEL = "logistic_regression"
FUTURE_MODEL_CANDIDATES = ["random_forest", "xgboost", "lightgbm"]
RANDOM_SEED = 42
USE_EMBEDDINGS = False

FEATURE_COLUMNS = [
    "facility_score",
    "contingency_score",
    "voltage_match",
    "voltage_near_match",
    "token_overlap",
    "zone_overlap",
    "interface_keyword_match",
    "embedding_similarity",
]

RULE_SCORE_V1_WEIGHTS = {
    "facility_score": 0.50,
    "contingency_score": 0.25,
    "voltage_match": 0.15,
    "zone_overlap": 0.10,
}

PANO_RULE_SCORE_V2_WEIGHTS = {
    "facility_score": 0.45,
    "contingency_score": 0.30,
    "voltage_match": 0.15,
    "token_overlap": 0.05,
    "zone_overlap": 0.05,
}

DAYZER_RULE_SCORE_V2_WEIGHTS = {
    "facility_score": 0.65,
    "voltage_match": 0.15,
    "token_overlap": 0.10,
    "interface_keyword_match": 0.10,
}

## Load Source Files

In [ ]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "Assignment 1 - constraint_mapping").exists():
            return candidate
    raise FileNotFoundError("Could not find data/Assignment 1 - constraint_mapping from the current path.")

repo_root = find_repo_root()
data_dir = repo_root / "data" / "Assignment 1 - constraint_mapping"
execution_outputs_dir = repo_root / "solution" / "assignment_1" / "execution_outputs"
execution_outputs_dir.mkdir(parents=True, exist_ok=True)

constraint_files = {
    "dayzer": data_dir / "Dayzer PJMISO constraint list.csv",
    "market": data_dir / "Market PJMISO constraint list.csv",
    "pano": data_dir / "Pano PJMISO constraint list.csv",
}

constraints = {name: pd.read_csv(path) for name, path in constraint_files.items()}
for name, df in constraints.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

## Normalize and Extract Features

Each source record is converted into comparable features: facility text, contingency text, voltage/kV, utility/zone tokens, facility tokens, contingency tokens, interface keywords, and extra tokens.

In [ ]:
SEPARATORS_PATTERN = re.compile(r"[-_./]+")
PUNCTUATION_PATTERN = re.compile(r"[^a-z0-9\s]")
VOLTAGE_PATTERN = re.compile(r"(?<!\d)(\d{2,4})\s*k\s*v\b|(?<!\d)(\d{2,4})kv\b")
STOP_TOKENS = {"", "a", "b", "c", "the", "and", "or", "to", "from", "kv", "line", "l"}
INTERFACE_KEYWORDS = {"interface", "import", "export", "flowgate", "flow", "tie", "path"}


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = SEPARATORS_PATTERN.sub(" ", text)
    text = re.sub(r"\bk\s*v\b", "kv", text)
    text = re.sub(r"\bxfmer\b", "transformer", text)
    text = re.sub(r"\bser\s+dev\b", "series device", text)
    text = PUNCTUATION_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_voltage(*values):
    text = normalize_text(" ".join("" if pd.isna(value) else str(value) for value in values))
    match = VOLTAGE_PATTERN.search(text)
    if not match:
        return pd.NA
    return int(next(group for group in match.groups() if group))


def tokenize(text):
    return [token for token in normalize_text(text).split() if token not in STOP_TOKENS and not token.isdigit()]


def unique_tokens(tokens):
    return sorted(set(tokens))


def extract_interface_keywords(tokens):
    return sorted(token for token in set(tokens) if token in INTERFACE_KEYWORDS or token.endswith("interface"))


def extract_utility_zone_tokens(zone_value, raw_text):
    tokens = []
    if pd.notna(zone_value) and str(zone_value).strip():
        tokens.extend(tokenize(zone_value))
    tokens.extend(token.lower() for token in re.findall(r"\b[A-Z]{2,}\b", str(raw_text)))
    return unique_tokens(tokens)


def build_feature_frame(dataset_name, df):
    records = []
    for source_index, row in df.iterrows():
        if dataset_name == "market":
            facility_raw = " | ".join(str(row.get(col, "")) for col in ["CONSTRAINT", "REPORTEDNAME"] if pd.notna(row.get(col, pd.NA)))
            contingency_raw = row.get("CONTINGENCY", "")
            zone_raw = row.get("TOZONE", pd.NA)
            display_constraint = row.get("CONSTRAINT", "")
            record_id = row.get("CONSTRAINTID", source_index)
        elif dataset_name == "pano":
            facility_raw = row.get("Monitored Facility", "")
            contingency_raw = row.get("Contingency Name", "")
            zone_raw = pd.NA
            display_constraint = row.get("Monitored Facility", "")
            record_id = row.get("PID", source_index)
        else:
            facility_raw = row.get("NAME", "")
            contingency_raw = ""
            zone_raw = pd.NA
            display_constraint = row.get("NAME", "")
            record_id = row.get("CID", source_index)

        facility_tokens = unique_tokens(tokenize(facility_raw))
        contingency_tokens = unique_tokens(tokenize(contingency_raw))
        combined_text = f"{facility_raw} {contingency_raw}"
        all_tokens = unique_tokens([*facility_tokens, *contingency_tokens])
        records.append(
            {
                "dataset": dataset_name,
                "record_id": record_id,
                "source_index": source_index,
                "display_constraint": display_constraint,
                "facility_raw": facility_raw,
                "contingency_raw": contingency_raw,
                "facility_normalized": normalize_text(facility_raw),
                "contingency_normalized": normalize_text(contingency_raw),
                "voltage_kv": extract_voltage(facility_raw, contingency_raw),
                "utility_zone_tokens": extract_utility_zone_tokens(zone_raw, combined_text),
                "facility_tokens": facility_tokens,
                "contingency_tokens": contingency_tokens,
                "interface_keywords": extract_interface_keywords(all_tokens),
                "extra_tokens": all_tokens,
            }
        )
    return pd.DataFrame(records)


feature_frames = {name: build_feature_frame(name, df) for name, df in constraints.items()}
parsed_features = pd.concat(feature_frames.values(), ignore_index=True)
parsed_features_export = parsed_features.copy()
for col in ["utility_zone_tokens", "facility_tokens", "contingency_tokens", "interface_keywords", "extra_tokens"]:
    parsed_features_export[col] = parsed_features_export[col].apply(lambda values: " | ".join(values))

advanced_features_path = execution_outputs_dir / "advanced_parsed_features.csv"
parsed_features_export.to_csv(advanced_features_path, index=False)
print(f"Saved parsed features to {advanced_features_path}")
parsed_features_export.head()

## Generate Top-K Candidate Pairs

Candidate generation uses token and voltage indexes to avoid comparing every record to every other record. The final candidate-pair output keeps the top 25 PANO and top 25 DAYZER candidates per PJMISO row.

`rule_score_v1` preserves the original shared formula. `rule_score_v2` is the accepted source-specific formula: PANO gives more weight to contingency and less to zone, while DAYZER relies mostly on facility/interface similarity because it lacks contingency detail. Downstream ranking uses `rule_score_v2`, while v1 remains available for comparison.

In [ ]:
def dice_similarity(left, right):
    left = set(left)
    right = set(right)
    if not left or not right:
        return 0.0
    return 2 * len(left & right) / (len(left) + len(right))


def voltage_match_score(left_voltage, right_voltage):
    if pd.isna(left_voltage) or pd.isna(right_voltage):
        return 0.0
    return 1.0 if int(left_voltage) == int(right_voltage) else 0.0


def voltage_near_match_score(left_voltage, right_voltage):
    if pd.isna(left_voltage) or pd.isna(right_voltage):
        return 0.0
    difference = abs(int(left_voltage) - int(right_voltage))
    if difference == 0:
        return 1.0
    return 1.0 if difference <= 5 else 0.0


def interface_keyword_score(left, right):
    left = set(left)
    right = set(right)
    if not left and not right:
        return 0.0
    return 1.0 if left & right else 0.0


def weighted_score(scores, weights):
    return sum(weights[feature] * scores.get(feature, 0.0) for feature in weights)


def score_candidate(reference, candidate, match_source):
    facility_score = dice_similarity(reference.facility_tokens, candidate.facility_tokens)
    contingency_score = dice_similarity(reference.contingency_tokens, candidate.contingency_tokens)
    voltage_match = voltage_match_score(reference.voltage_kv, candidate.voltage_kv)
    voltage_near_match = voltage_near_match_score(reference.voltage_kv, candidate.voltage_kv)
    token_overlap = dice_similarity(reference.extra_tokens, candidate.extra_tokens)
    zone_overlap = dice_similarity(reference.utility_zone_tokens, candidate.utility_zone_tokens)
    interface_match = interface_keyword_score(reference.interface_keywords, candidate.interface_keywords)
    embedding_similarity = pd.NA if not USE_EMBEDDINGS else pd.NA
    scores = {
        "facility_score": facility_score,
        "contingency_score": contingency_score,
        "voltage_match": voltage_match,
        "voltage_near_match": voltage_near_match,
        "token_overlap": token_overlap,
        "zone_overlap": zone_overlap,
        "interface_keyword_match": interface_match,
        "embedding_similarity": embedding_similarity,
    }
    scores["rule_score_v1"] = weighted_score(scores, RULE_SCORE_V1_WEIGHTS)
    if match_source == "pano":
        scores["rule_score_v2"] = weighted_score(scores, PANO_RULE_SCORE_V2_WEIGHTS)
    else:
        scores["rule_score_v2"] = weighted_score(scores, DAYZER_RULE_SCORE_V2_WEIGHTS)
    scores["rule_score"] = scores["rule_score_v2"]
    return scores


def build_candidate_index(target_features, include_contingency=True):
    token_index = defaultdict(set)
    voltage_index = defaultdict(set)
    for row in target_features.itertuples():
        tokens = set(row.facility_tokens) | set(row.interface_keywords)
        if include_contingency:
            tokens |= set(row.contingency_tokens)
        for token in tokens:
            token_index[token].add(row.Index)
        if pd.notna(row.voltage_kv):
            voltage_index[int(row.voltage_kv)].add(row.Index)
    return token_index, voltage_index


def coarse_candidate_indices(reference, token_index, voltage_index, include_contingency=True):
    tokens = set(reference.facility_tokens) | set(reference.interface_keywords)
    if include_contingency:
        tokens |= set(reference.contingency_tokens)

    candidate_counts = defaultdict(int)
    for token in tokens:
        for idx in token_index.get(token, set()):
            candidate_counts[idx] += 1

    if pd.notna(reference.voltage_kv):
        for idx in voltage_index.get(int(reference.voltage_kv), set()):
            candidate_counts[idx] += 1

    ranked = sorted(candidate_counts, key=candidate_counts.get, reverse=True)
    return ranked[:MAX_CANDIDATES_TO_SCORE]


def generate_candidate_pairs(reference_features, target_features, match_source, include_contingency=True):
    token_index, voltage_index = build_candidate_index(target_features, include_contingency=include_contingency)
    rows = []
    for reference in reference_features.itertuples():
        candidates = coarse_candidate_indices(reference, token_index, voltage_index, include_contingency=include_contingency)
        scored_candidates = []
        for idx in candidates:
            candidate = target_features.loc[idx]
            scores = score_candidate(reference, candidate, match_source)
            scored_candidates.append((scores["rule_score"], idx, candidate, scores))

        scored_candidates.sort(key=lambda item: item[0], reverse=True)
        for rank, (_, idx, candidate, scores) in enumerate(scored_candidates[:TOP_K_PER_SOURCE], start=1):
            rows.append(
                {
                    "market_record_id": reference.record_id,
                    "market_index": reference.source_index,
                    "market_constraint": reference.display_constraint,
                    "match_source": match_source,
                    "candidate_record_id": candidate.record_id,
                    "candidate_index": candidate.source_index,
                    "candidate_constraint": candidate.display_constraint,
                    "candidate_rank_initial": rank,
                    **{key: round(value, 6) if isinstance(value, float) else value for key, value in scores.items()},
                }
            )
    return pd.DataFrame(rows)


market_features = feature_frames["market"].copy()
pano_features = feature_frames["pano"].copy()
dayzer_features = feature_frames["dayzer"].copy()

pano_candidate_pairs = generate_candidate_pairs(market_features, pano_features, "pano", include_contingency=True)
dayzer_candidate_pairs = generate_candidate_pairs(market_features, dayzer_features, "dayzer", include_contingency=False)
advanced_candidate_pairs = pd.concat([pano_candidate_pairs, dayzer_candidate_pairs], ignore_index=True)
advanced_candidate_pairs_path = execution_outputs_dir / "advanced_candidate_pairs.csv"
advanced_candidate_pairs.to_csv(advanced_candidate_pairs_path, index=False)
print(f"Saved advanced candidate pairs to {advanced_candidate_pairs_path}")
print(advanced_candidate_pairs.groupby("match_source").size())
advanced_candidate_pairs.head()

## Add Ambiguity and Review Flags

In [ ]:
def add_group_scoring_flags(candidate_pairs):
    df = candidate_pairs.copy()
    df = df.sort_values(["market_index", "match_source", "rule_score"], ascending=[True, True, False])
    df["candidate_rank"] = df.groupby(["market_index", "match_source"]).cumcount() + 1
    df["best_score"] = df.groupby(["market_index", "match_source"])["rule_score"].transform("max")

    second_scores = (
        df[df["candidate_rank"] == 2][["market_index", "match_source", "rule_score"]]
        .rename(columns={"rule_score": "second_best_score"})
    )
    df = df.merge(second_scores, on=["market_index", "match_source"], how="left")
    df["second_best_score"] = df["second_best_score"].fillna(0.0)
    df["score_margin"] = df["best_score"] - df["second_best_score"]
    df["ambiguity_flag"] = (df["best_score"] < LOW_SCORE_REVIEW_THRESHOLD) | (df["score_margin"] < CLOSE_MARGIN_THRESHOLD)
    df["low_score_flag"] = df["rule_score"] < LOW_SCORE_REVIEW_THRESHOLD
    df["close_margin_flag"] = df["score_margin"] < CLOSE_MARGIN_THRESHOLD
    return df


advanced_candidate_pairs = add_group_scoring_flags(advanced_candidate_pairs)
advanced_candidate_pairs.to_csv(advanced_candidate_pairs_path, index=False)
advanced_candidate_pairs[["market_constraint", "match_source", "candidate_constraint", "candidate_rank", "rule_score", "best_score", "second_best_score", "ambiguity_flag"]].head(10)

## Build Strategic Review Queue and Manual Label Template

The review queue includes selected matches that are ambiguous, low-scoring, or close-margin, plus a random audit sample of high-confidence top-ranked matches.

The separate random hand-grading file follows the original manual-labeling plan: 100 high-confidence, 100 medium-confidence, and 100 low/ambiguous candidate pairs.

In [ ]:
top_ranked = advanced_candidate_pairs[advanced_candidate_pairs["candidate_rank"] == 1].copy()
flagged_review = top_ranked[
    top_ranked["ambiguity_flag"]
    | top_ranked["low_score_flag"]
    | top_ranked["close_margin_flag"]
].copy()

audit_pool = top_ranked[
    (~top_ranked["ambiguity_flag"])
    & (top_ranked["rule_score"] >= HIGH_CONFIDENCE_THRESHOLD)
].copy()
audit_sample = audit_pool.sample(n=min(RANDOM_AUDIT_SAMPLE_SIZE, len(audit_pool)), random_state=RANDOM_SEED) if len(audit_pool) else audit_pool

advanced_review_queue = (
    pd.concat([flagged_review, audit_sample], ignore_index=True)
    .drop_duplicates(subset=["market_index", "match_source", "candidate_index"])
    .sort_values(["ambiguity_flag", "low_score_flag", "close_margin_flag", "market_index", "match_source", "candidate_rank"], ascending=[False, False, False, True, True, True])
)

advanced_review_queue_path = execution_outputs_dir / "advanced_review_queue.csv"
advanced_review_queue.to_csv(advanced_review_queue_path, index=False)

manual_label_columns = [
    "market_record_id",
    "market_index",
    "market_constraint",
    "match_source",
    "candidate_record_id",
    "candidate_index",
    "candidate_constraint",
    *FEATURE_COLUMNS,
    "rule_score",
    "rule_score_v1",
    "rule_score_v2",
    "best_score",
    "second_best_score",
    "score_margin",
    "ambiguity_flag",
    "manual_label",
    "review_notes",
]
manual_labels_template = advanced_review_queue.copy()
manual_labels_template["manual_label"] = ""
manual_labels_template["review_notes"] = ""
manual_labels_template = manual_labels_template[manual_label_columns]
manual_labels_template_path = execution_outputs_dir / "manual_labels_template.csv"
manual_labels_template.to_csv(manual_labels_template_path, index=False)

print(f"Saved advanced review queue to {advanced_review_queue_path}")
print(f"Saved manual labels template to {manual_labels_template_path}")
print("review rows", len(advanced_review_queue))

high_confidence_pool = top_ranked[
    (~top_ranked["ambiguity_flag"])
    & (top_ranked["rule_score"] >= HIGH_CONFIDENCE_THRESHOLD)
].copy()
medium_confidence_pool = top_ranked[
    (top_ranked["rule_score"] >= MEDIUM_CONFIDENCE_THRESHOLD)
    & (top_ranked["rule_score"] < HIGH_CONFIDENCE_THRESHOLD)
].copy()
low_ambiguous_pool = top_ranked[
    (top_ranked["ambiguity_flag"])
    | (top_ranked["rule_score"] < MEDIUM_CONFIDENCE_THRESHOLD)
    | (top_ranked["score_margin"] < CLOSE_MARGIN_THRESHOLD)
].copy()


def sample_label_bucket(df, bucket_name, sample_size=MANUAL_LABEL_SAMPLE_SIZE_PER_BUCKET):
    sample = df.sample(n=min(sample_size, len(df)), random_state=RANDOM_SEED).copy() if len(df) else df.copy()
    sample["label_bucket"] = bucket_name
    return sample

manual_label_random_selection = pd.concat(
    [
        sample_label_bucket(high_confidence_pool, "high_confidence"),
        sample_label_bucket(medium_confidence_pool, "medium_confidence"),
        sample_label_bucket(low_ambiguous_pool, "low_or_ambiguous"),
    ],
    ignore_index=True,
)
manual_label_random_selection["manual_label"] = ""
manual_label_random_selection["review_notes"] = ""
manual_label_random_selection_columns = [
    "label_bucket",
    *manual_label_columns,
]
manual_label_random_selection = manual_label_random_selection[manual_label_random_selection_columns]
manual_label_random_selection_path = execution_outputs_dir / "manual_labels_random_selection.csv"
manual_label_random_selection.to_csv(manual_label_random_selection_path, index=False)
print(f"Saved manual label random selection to {manual_label_random_selection_path}")
print(manual_label_random_selection["label_bucket"].value_counts().to_string())

advanced_review_queue.head()

## Optional Weak-Supervision Classifier

If `manual_labels.csv` exists and contains at least 100 `match` and 100 `not_match` rows, train a class-balanced logistic regression model. Otherwise, use the transparent rule score as the final score. Logistic regression is the first model because it is interpretable; Random Forest, XGBoost, or LightGBM can be compared later if logistic regression metrics are weak after enough reliable labels exist.

In [ ]:
manual_labels_path = execution_outputs_dir / "manual_labels.csv"
model_trained = False
metrics = {}

candidate_predictions = advanced_candidate_pairs.copy()
candidate_predictions["model_probability"] = pd.NA

if manual_labels_path.exists():
    manual_labels = pd.read_csv(manual_labels_path)
    labeled = manual_labels[manual_labels["manual_label"].isin(["match", "not_match"])].copy()
    label_counts = labeled["manual_label"].value_counts()
    enough_labels = label_counts.get("match", 0) >= MIN_LABELS_PER_CLASS and label_counts.get("not_match", 0) >= MIN_LABELS_PER_CLASS
else:
    manual_labels = pd.DataFrame()
    labeled = pd.DataFrame()
    label_counts = pd.Series(dtype=int)
    enough_labels = False

if enough_labels:
    train_df = labeled.copy()
    train_df["label"] = (train_df["manual_label"] == "match").astype(int)
    X = train_df[FEATURE_COLUMNS].fillna(0.0)
    y = train_df["label"]
    stratify = y if y.nunique() == 2 else None
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=stratify)
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED),
    )
    classifier.fit(X_train, y_train)
    probabilities = classifier.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)
    metrics = {
        "trained": True,
        "model": CLASSIFIER_MODEL,
        "label_count": len(train_df),
        "match_count": int((y == 1).sum()),
        "not_match_count": int((y == 0).sum()),
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities) if y_test.nunique() == 2 else pd.NA,
    }
    candidate_predictions["model_probability"] = classifier.predict_proba(candidate_predictions[FEATURE_COLUMNS].fillna(0.0))[:, 1]
    model_trained = True
    metrics_path = execution_outputs_dir / "match_classifier_metrics.csv"
    pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
    print(f"Saved classifier metrics to {metrics_path}")
else:
    print(f"Insufficient manual labels for classifier training; need at least {MIN_LABELS_PER_CLASS} match and {MIN_LABELS_PER_CLASS} not_match labels. Using rule_score as final score.")
    print(label_counts.to_string() if len(label_counts) else "No manual_labels.csv found yet.")

candidate_predictions["model_probability"] = pd.to_numeric(candidate_predictions["model_probability"], errors="coerce")
if model_trained:
    candidate_predictions["final_score"] = candidate_predictions["model_probability"]
else:
    candidate_predictions["final_score"] = candidate_predictions["rule_score"].astype(float)
trained_predictions_path = execution_outputs_dir / "trained_match_predictions.csv"
candidate_predictions.to_csv(trained_predictions_path, index=False)
print(f"Saved candidate-level predictions to {trained_predictions_path}")
candidate_predictions.head()

## Select Final Refined Matches

In [ ]:
def confidence_label(score):
    if pd.isna(score):
        return "low"
    if score >= HIGH_CONFIDENCE_THRESHOLD:
        return "high"
    if score >= MEDIUM_CONFIDENCE_THRESHOLD:
        return "medium"
    return "low"


def best_predictions_by_source(predictions):
    ranked = predictions.sort_values(["market_index", "match_source", "final_score"], ascending=[True, True, False]).copy()
    ranked["final_rank"] = ranked.groupby(["market_index", "match_source"]).cumcount() + 1
    best = ranked[ranked["final_rank"] == 1].copy()
    best["accepted_match"] = best["final_score"] >= MIN_ACCEPT_SCORE
    best["selected_constraint"] = np.where(best["accepted_match"], best["candidate_constraint"], pd.NA)
    best["confidence"] = best["final_score"].apply(confidence_label)
    return best


best_by_source = best_predictions_by_source(candidate_predictions)
market_reference = market_features[["source_index", "record_id", "display_constraint"]].rename(
    columns={"source_index": "market_index", "record_id": "market_record_id", "display_constraint": "market_constraint"}
)

expected_match_sources = pd.MultiIndex.from_product(
    [market_reference["market_index"], ["pano", "dayzer"]],
    names=["market_index", "match_source"],
).to_frame(index=False)
best_by_source = expected_match_sources.merge(best_by_source, on=["market_index", "match_source"], how="left")
best_by_source = best_by_source.merge(market_reference, on="market_index", how="left", suffixes=("", "_reference"))
best_by_source["market_constraint"] = best_by_source["market_constraint"].fillna(best_by_source["market_constraint_reference"])
best_by_source = best_by_source.drop(columns=[col for col in ["market_constraint_reference"] if col in best_by_source.columns])
for col in ["facility_score", "contingency_score", "voltage_match", "voltage_near_match", "token_overlap", "zone_overlap", "interface_keyword_match", "rule_score", "best_score", "second_best_score", "score_margin", "final_score"]:
    best_by_source[col] = best_by_source[col].fillna(0.0)
for col in ["ambiguity_flag", "low_score_flag", "close_margin_flag"]:
    best_by_source[col] = best_by_source[col].where(best_by_source[col].notna(), True).astype(bool)
best_by_source["accepted_match"] = best_by_source["accepted_match"].where(best_by_source["accepted_match"].notna(), False).astype(bool)
best_by_source["confidence"] = best_by_source["confidence"].fillna("low")

pano_best = best_by_source[best_by_source["match_source"] == "pano"].copy()
dayzer_best = best_by_source[best_by_source["match_source"] == "dayzer"].copy()

constraint_matches_refined = market_reference.copy()
constraint_matches_refined = constraint_matches_refined.merge(
    dayzer_best[["market_index", "selected_constraint", "confidence", "final_score", "ambiguity_flag"]].rename(
        columns={
            "selected_constraint": "dayzer_constraint",
            "confidence": "dayzer_confidence",
            "final_score": "dayzer_final_score",
            "ambiguity_flag": "dayzer_ambiguity_flag",
        }
    ),
    on="market_index",
    how="left",
)
constraint_matches_refined = constraint_matches_refined.merge(
    pano_best[["market_index", "selected_constraint", "confidence", "final_score", "ambiguity_flag"]].rename(
        columns={
            "selected_constraint": "pano_constraint",
            "confidence": "pano_confidence",
            "final_score": "pano_final_score",
            "ambiguity_flag": "pano_ambiguity_flag",
        }
    ),
    on="market_index",
    how="left",
)

support_columns = [
    "market_constraint",
    "match_source",
    "candidate_constraint",
    "accepted_match",
    "confidence",
    "final_score",
    "model_probability",
    "rule_score",
    "rule_score_v1",
    "rule_score_v2",
    "facility_score",
    "contingency_score",
    "voltage_match",
    "voltage_near_match",
    "token_overlap",
    "zone_overlap",
    "interface_keyword_match",
    "embedding_similarity",
    "best_score",
    "second_best_score",
    "score_margin",
    "ambiguity_flag",
    "low_score_flag",
    "close_margin_flag",
]
constraint_match_support_refined = best_by_source[support_columns].copy()

matches_refined_path = execution_outputs_dir / "constraint_matches_refined.csv"
support_refined_path = execution_outputs_dir / "constraint_match_support_refined.csv"
constraint_matches_refined.to_csv(matches_refined_path, index=False)
constraint_match_support_refined.to_csv(support_refined_path, index=False)
print(f"Saved refined matches to {matches_refined_path}")
print(f"Saved refined support rows to {support_refined_path}")
constraint_matches_refined.head()

## Validation Checks

In [ ]:
validation_results = {
    "pjmiso_rows": len(market_features),
    "refined_match_rows": len(constraint_matches_refined),
    "support_rows": len(constraint_match_support_refined),
    "expected_support_rows": len(market_features) * 2,
    "every_pjmiso_row_present": len(constraint_matches_refined) == len(market_features),
    "every_pjmiso_source_pair_present": len(constraint_match_support_refined) == len(market_features) * 2,
    "below_threshold_matches_accepted": int((constraint_match_support_refined["accepted_match"] & (constraint_match_support_refined["final_score"] < MIN_ACCEPT_SCORE)).sum()),
    "review_queue_rows": len(advanced_review_queue),
    "flagged_rows_missing_from_review_queue": 0,
}

top_ranked_for_validation = advanced_candidate_pairs[advanced_candidate_pairs["candidate_rank"] == 1]
flagged_keys = top_ranked_for_validation[
    top_ranked_for_validation["ambiguity_flag"]
    | top_ranked_for_validation["low_score_flag"]
    | top_ranked_for_validation["close_margin_flag"]
][["market_index", "match_source", "candidate_index"]].drop_duplicates()
review_keys = advanced_review_queue[["market_index", "match_source", "candidate_index"]].drop_duplicates()
missing_flagged = flagged_keys.merge(review_keys, on=["market_index", "match_source", "candidate_index"], how="left", indicator=True)
validation_results["flagged_rows_missing_from_review_queue"] = int((missing_flagged["_merge"] == "left_only").sum())

validation_path = execution_outputs_dir / "advanced_matching_validation_summary.csv"
pd.DataFrame([validation_results]).to_csv(validation_path, index=False)
print(f"Saved validation summary to {validation_path}")
validation_results

## Human-in-the-Loop Supervised Refinement

The original rule-based matcher is used for candidate generation. Manual labels are used only for supervised confidence refinement.

Label convention:

- `1` = match
- `0` = unmatch / not a valid match
- `-1` = uncertain / cannot determine

Rows labeled `-1` are excluded from training. The final classifier probability is used as a calibrated confidence estimate, not as external ground truth. Original heuristic score columns are preserved for auditability.

In [ ]:
reviewed_labels_path = execution_outputs_dir / "manual_labels_random_selection_reviewed.csv"
if not reviewed_labels_path.exists():
    raise FileNotFoundError(f"Reviewed manual label file not found: {reviewed_labels_path}")

reviewed_labels = pd.read_csv(reviewed_labels_path)
reviewed_labels["manual_label"] = pd.to_numeric(reviewed_labels["manual_label"], errors="coerce")
training_labels = reviewed_labels[reviewed_labels["manual_label"].isin([0, 1])].copy()

candidate_pool = advanced_candidate_pairs.copy()
candidate_pool["zone_or_token_overlap"] = candidate_pool[["zone_overlap", "token_overlap"]].max(axis=1)
candidate_pool["weighted_score"] = candidate_pool["rule_score"]
candidate_pool["score_gap_to_second_best"] = candidate_pool["score_margin"]
candidate_pool["ambiguity_flag_numeric"] = candidate_pool["ambiguity_flag"].astype(int)
candidate_pool["confidence_label"] = pd.cut(
    candidate_pool["rule_score"],
    bins=[-np.inf, 0.50, 0.75, 0.90, np.inf],
    labels=["reject", "uncertain", "medium_high", "high"],
)
confidence_encoding = {"reject": 0, "uncertain": 1, "medium_high": 2, "high": 3}
candidate_pool["confidence_label_encoded"] = candidate_pool["confidence_label"].map(confidence_encoding).astype(float)

supervised_feature_columns = [
    "weighted_score",
    "rule_score_v1",
    "rule_score_v2",
    "facility_score",
    "contingency_score",
    "voltage_match",
    "voltage_near_match",
    "zone_or_token_overlap",
    "interface_keyword_match",
    "ambiguity_flag_numeric",
    "confidence_label_encoded",
    "score_gap_to_second_best",
]

label_keys = ["market_index", "match_source", "candidate_index"]
training_data = training_labels.merge(
    candidate_pool[label_keys + supervised_feature_columns],
    on=label_keys,
    how="left",
    suffixes=("", "_candidate"),
)

missing_feature_rows = training_data[supervised_feature_columns].isna().all(axis=1).sum()
training_data = training_data.dropna(subset=supervised_feature_columns, how="all").copy()
for col in supervised_feature_columns:
    training_data[col] = pd.to_numeric(training_data[col], errors="coerce").fillna(0.0)

X = training_data[supervised_feature_columns]
y = training_data["manual_label"].astype(int)
positive_count = int((y == 1).sum())
negative_count = int((y == 0).sum())
print(f"Reviewed rows: {len(reviewed_labels):,}")
print(f"Training rows after excluding -1: {len(training_data):,}")
print(f"Positive labels: {positive_count:,}; negative labels: {negative_count:,}; missing feature rows dropped: {missing_feature_rows:,}")
training_data.head()

## Train Supervised Confidence Model

Logistic Regression is attempted first because it is interpretable. If it cannot train because the labeled feature matrix is degenerate, the workflow falls back to Random Forest.

In [ ]:
def refined_confidence_label(probability):
    if probability >= 0.90:
        return "high"
    if probability >= 0.75:
        return "medium_high"
    if probability >= 0.50:
        return "uncertain"
    return "reject"


def build_logistic_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_SEED),
    )


def build_random_forest_model():
    return RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )

model_name = pd.NA
supervised_model = None
training_error = ""
metrics_records = []
coefficient_frame = pd.DataFrame()
feature_importance_frame = pd.DataFrame()

if y.nunique() < 2:
    training_error = "Need both positive and negative labels to train."
else:
    try:
        non_constant_features = X.columns[X.nunique(dropna=False) > 1].tolist()
        if not non_constant_features:
            raise ValueError("All supervised features are constant.")
        X_trainable = X[non_constant_features]
        supervised_model = build_logistic_model()
        supervised_model.fit(X_trainable, y)
        model_name = "LogisticRegression"
        logistic = supervised_model.named_steps["logisticregression"]
        coefficient_frame = pd.DataFrame(
            {
                "model": model_name,
                "feature": non_constant_features,
                "coefficient": logistic.coef_[0],
            }
        ).sort_values("coefficient", ascending=False)
    except Exception as exc:
        training_error = f"LogisticRegression failed: {exc}. Falling back to RandomForestClassifier."
        non_constant_features = X.columns[X.nunique(dropna=False) > 1].tolist()
        X_trainable = X[non_constant_features]
        supervised_model = build_random_forest_model()
        supervised_model.fit(X_trainable, y)
        model_name = "RandomForestClassifier"
        feature_importance_frame = pd.DataFrame(
            {
                "model": model_name,
                "feature": non_constant_features,
                "importance": supervised_model.feature_importances_,
            }
        ).sort_values("importance", ascending=False)

if supervised_model is None:
    print(training_error)
else:
    min_class_count = min(positive_count, negative_count)
    if min_class_count >= 3:
        cv_splits = min(5, min_class_count)
        cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_SEED)
        scoring = ["accuracy", "precision", "recall", "f1"]
        cv_results = cross_validate(supervised_model, X_trainable, y, cv=cv, scoring=scoring)
        for metric in scoring:
            values = cv_results[f"test_{metric}"]
            metrics_records.append({"metric": f"cv_{metric}_mean", "value": float(values.mean())})
            metrics_records.append({"metric": f"cv_{metric}_std", "value": float(values.std())})
    else:
        metrics_records.append({"metric": "cv_status", "value": "Not enough labels per class for cross-validation"})

    metrics_records.extend(
        [
            {"metric": "model", "value": model_name},
            {"metric": "labeled_rows_used", "value": int(len(training_data))},
            {"metric": "positive_labels", "value": positive_count},
            {"metric": "negative_labels", "value": negative_count},
            {"metric": "excluded_uncertain_labels", "value": int((reviewed_labels["manual_label"] == -1).sum())},
            {"metric": "training_error_or_fallback", "value": training_error},
        ]
    )

supervised_diagnostics = pd.DataFrame(metrics_records)
supervised_diagnostics_path = execution_outputs_dir / "supervised_refinement_diagnostics.csv"
supervised_diagnostics.to_csv(supervised_diagnostics_path, index=False)

if len(coefficient_frame):
    coefficient_path = execution_outputs_dir / "supervised_logistic_coefficients.csv"
    coefficient_frame.to_csv(coefficient_path, index=False)
if len(feature_importance_frame):
    importance_path = execution_outputs_dir / "supervised_random_forest_feature_importance.csv"
    feature_importance_frame.to_csv(importance_path, index=False)

print(f"Saved supervised diagnostics to {supervised_diagnostics_path}")
print(supervised_diagnostics.to_string(index=False))
if len(coefficient_frame):
    display(coefficient_frame)
if len(feature_importance_frame):
    display(feature_importance_frame)

## Predict Supervised Match Probability and Re-Rank Candidates

The classifier predicts `match_probability` for every generated candidate pair. PANO and DAYZER candidates are re-ranked separately for each PJMISO market constraint. If the highest probability for a source is below 0.50, that source is left unmatched.

In [ ]:
supervised_predictions = candidate_pool.copy()
for col in supervised_feature_columns:
    supervised_predictions[col] = pd.to_numeric(supervised_predictions[col], errors="coerce").fillna(0.0)

if supervised_model is None:
    supervised_predictions["match_probability"] = supervised_predictions["rule_score"].astype(float)
else:
    supervised_predictions["match_probability"] = supervised_model.predict_proba(supervised_predictions[non_constant_features])[:, 1]

supervised_predictions["refined_confidence_label"] = supervised_predictions["match_probability"].apply(refined_confidence_label)
manual_label_lookup = reviewed_labels[label_keys + ["manual_label"]].drop_duplicates(label_keys)
supervised_predictions = supervised_predictions.merge(manual_label_lookup, on=label_keys, how="left")
supervised_predictions["original_weighted_score"] = supervised_predictions["rule_score"]

supervised_predictions_path = execution_outputs_dir / "supervised_candidate_predictions.csv"
supervised_predictions.to_csv(supervised_predictions_path, index=False)
print(f"Saved supervised candidate predictions to {supervised_predictions_path}")
supervised_predictions[["market_constraint", "match_source", "candidate_constraint", "original_weighted_score", "match_probability", "refined_confidence_label", "manual_label"]].head()

In [ ]:
ranked_supervised = supervised_predictions.sort_values(
    ["market_index", "match_source", "match_probability"],
    ascending=[True, True, False],
).copy()
ranked_supervised["supervised_rank"] = ranked_supervised.groupby(["market_index", "match_source"]).cumcount() + 1
best_supervised = ranked_supervised[ranked_supervised["supervised_rank"] == 1].copy()
best_supervised["accepted_supervised_match"] = best_supervised["match_probability"] >= 0.50
best_supervised["selected_constraint"] = np.where(best_supervised["accepted_supervised_match"], best_supervised["candidate_constraint"], pd.NA)

market_reference = market_features[["source_index", "display_constraint"]].rename(
    columns={"source_index": "market_index", "display_constraint": "market_constraint"}
)
source_grid = pd.MultiIndex.from_product(
    [market_reference["market_index"], ["dayzer", "pano"]],
    names=["market_index", "match_source"],
).to_frame(index=False)
best_supervised = source_grid.merge(best_supervised, on=["market_index", "match_source"], how="left")
best_supervised = best_supervised.merge(market_reference, on="market_index", how="left", suffixes=("", "_reference"))
best_supervised["market_constraint"] = best_supervised["market_constraint"].fillna(best_supervised["market_constraint_reference"])

supervised_dayzer = best_supervised[best_supervised["match_source"] == "dayzer"][["market_index", "selected_constraint"]].rename(columns={"selected_constraint": "dayzer_constraint"})
supervised_pano = best_supervised[best_supervised["match_source"] == "pano"][["market_index", "selected_constraint"]].rename(columns={"selected_constraint": "pano_constraint"})

supervised_constraint_matches_refined = market_reference.merge(supervised_dayzer, on="market_index", how="left").merge(supervised_pano, on="market_index", how="left")
supervised_constraint_matches_refined = supervised_constraint_matches_refined[["market_constraint", "dayzer_constraint", "pano_constraint"]]

supervised_support = best_supervised.rename(
    columns={
        "match_source": "candidate_source",
    }
)
if "original_weighted_score" not in supervised_support.columns:
    supervised_support["original_weighted_score"] = supervised_support["rule_score"]
for col in ["candidate_constraint", "match_probability", "refined_confidence_label", "facility_score", "contingency_score", "voltage_match", "zone_or_token_overlap", "ambiguity_flag", "manual_label"]:
    if col not in supervised_support.columns:
        supervised_support[col] = pd.NA
supervised_constraint_match_support_refined = supervised_support[
    [
        "market_constraint",
        "candidate_constraint",
        "candidate_source",
        "original_weighted_score",
        "match_probability",
        "refined_confidence_label",
        "facility_score",
        "contingency_score",
        "voltage_match",
        "zone_or_token_overlap",
        "ambiguity_flag",
        "manual_label",
    ]
].copy()

supervised_matches_path = execution_outputs_dir / "supervised_constraint_matches_refined.csv"
supervised_support_path = execution_outputs_dir / "supervised_constraint_match_support_refined.csv"
supervised_constraint_matches_refined.to_csv(supervised_matches_path, index=False)
supervised_constraint_match_support_refined.to_csv(supervised_support_path, index=False)
print(f"Saved supervised refined matches to {supervised_matches_path}")
print(f"Saved supervised refined support to {supervised_support_path}")
supervised_constraint_matches_refined.head()

## Supervised Refinement Validation

In [ ]:
supervised_validation = {
    "reviewed_rows": int(len(reviewed_labels)),
    "labeled_rows_used": int(len(training_data)),
    "positive_labels": positive_count,
    "negative_labels": negative_count,
    "uncertain_labels_excluded": int((reviewed_labels["manual_label"] == -1).sum()),
    "model": str(model_name),
    "supervised_match_rows": int(len(supervised_constraint_matches_refined)),
    "supervised_support_rows": int(len(supervised_constraint_match_support_refined)),
    "every_pjmiso_row_present": bool(len(supervised_constraint_matches_refined) == len(market_features)),
    "expected_support_rows_present": bool(len(supervised_constraint_match_support_refined) == len(market_features) * 2),
    "accepted_below_050": int((best_supervised["accepted_supervised_match"].astype(bool) & (best_supervised["match_probability"].fillna(0) < 0.50)).sum()),
}

supervised_validation_path = execution_outputs_dir / "supervised_refinement_validation_summary.csv"
pd.DataFrame([supervised_validation]).to_csv(supervised_validation_path, index=False)
print(f"Saved supervised validation summary to {supervised_validation_path}")
supervised_validation